# Simple Models – Unified Training

Train CNN, RNN, or LSTM. **Plug and play**: set `MODEL` below. LM fusion for all models.

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from pretraining import load_preprocessed_dataset, get_class_weights
from src.Constants.char_to_key import CHAR_TO_INDEX, NUM_CLASSES, INDEX_TO_CHAR
from src.decoding.lm_fusion import build_char_ngram_lm, build_interpolated_char_lm
from ml.models.loss_functions.custom_losses import FocalLoss
from src.visualisation.visualisation import compute_confusion_matrix_40x40, plot_virtual_keyboard_heatmap

from simple_models import (
    SimpleCNNClassifier,
    SimpleRNNClassifier,
    SimpleLSTMClassifier,
    fuse_batch_logits_with_lm,
    extract_prev_idx,
)

torch.manual_seed(42)
np.random.seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Device: mps


## Config 

In [2]:
# Set model: "cnn" | "rnn" | "lstm"
MODEL = "cnn"

DATA_PATH = "data_jun_4/processed_dataset.pt"
BATCH_SIZE = 32
EPOCHS = 200
EARLY_STOPPING_PATIENCE = 30  # stop if no val improvement for this many epochs

LR = 1e-3
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15

# LM fusion (all models)
LM_BETA = 0.5
LM_AUX_WEIGHT = 1
LM_ORDER = 4
LM_USE_INTERPOLATED = True
LM_ADD_K = 0.05

In [3]:
dataset = load_preprocessed_dataset(
    DATA_PATH,
    char_to_index=CHAR_TO_INDEX,
    is_one_hot_labels=False,
    return_class_id=False,
    add_prev_char=True,
)
n = len(dataset)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)
train_ds = Subset(dataset, range(0, n_train))
val_ds = Subset(dataset, range(n_train, n_train + n_val))
test_ds = Subset(dataset, range(n_train + n_val, n))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"{MODEL.upper()}: {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test")

CNN: 4165 train, 892 val, 894 test


In [4]:
# Plug and play: pick model
MODEL_CLS = {"cnn": SimpleCNNClassifier, "rnn": SimpleRNNClassifier, "lstm": SimpleLSTMClassifier}[MODEL]
KWARGS = {
    "cnn": {"hidden_dim": 64, "kernel_sizes": (3, 5, 7), "dropout": 0.2},
    "rnn": {"hidden_dim": 64, "num_layers": 1, "dropout": 0.2},
    "lstm": {"hidden_dim": 64, "num_layers": 1, "dropout": 0.2},
}[MODEL]
class_weights = get_class_weights(DATA_PATH, train_ratio=0.8, split_strategy="contiguous").to(DEVICE)
criterion = FocalLoss(gamma=2.0, alpha=class_weights)
model = MODEL_CLS(input_dim=dataset.input_dim, num_classes=NUM_CLASSES, **KWARGS).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
idx_to_char = {i: INDEX_TO_CHAR[i] for i in range(NUM_CLASSES)}

# Print model architecture (torchinfo if available, else same format as train_model.ipynb)
try:
    from torchinfo import summary
    x_sample = dataset[0][0]
    input_shape = (1, *x_sample.shape)  # (B, T, D)
    summary(model, input_shape, col_names=["input_size", "output_size", "num_params"], verbose=1)
    model.to(DEVICE)  # torchinfo can move model to CPU; ensure it stays on DEVICE for training
except ImportError:
    print("\n" + "=" * 60)
    print("Model architecture")
    print("=" * 60)
    print(model)
    print("=" * 60)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# LM fusion for all models
train_chars = [dataset._labels[i] for i in range(n_train)]
char_lm = build_interpolated_char_lm(train_chars, max_order=LM_ORDER, add_k=LM_ADD_K) if LM_USE_INTERPOLATED else build_char_ngram_lm(train_chars, order=LM_ORDER, add_k=LM_ADD_K)
print("LM fusion enabled")

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
SimpleCNNClassifier                      [1, 12, 158]              [1, 40]                   --
├─ModuleList: 1-1                        --                        --                        --
│    └─Sequential: 2-1                   [1, 158, 12]              [1, 64, 12]               --
│    │    └─Conv1d: 3-1                  [1, 158, 12]              [1, 64, 12]               30,400
│    │    └─BatchNorm1d: 3-2             [1, 64, 12]               [1, 64, 12]               128
│    │    └─ReLU: 3-3                    [1, 64, 12]               [1, 64, 12]               --
│    │    └─Dropout: 3-4                 [1, 64, 12]               [1, 64, 12]               --
│    └─Sequential: 2-2                   [1, 158, 12]              [1, 64, 12]               --
│    │    └─Conv1d: 3-5                  [1, 158, 12]              [1, 64, 12]               50,624
│    │    └─BatchNorm1d: 3

In [ ]:
def train_epoch():
    model.train()
    total = 0.0
    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        if LM_BETA > 0 and char_lm is not None:
            prev_idx = extract_prev_idx(x, NUM_CLASSES)
            if prev_idx is not None:
                fused = fuse_batch_logits_with_lm(logits, char_lm, prev_idx, LM_BETA, idx_to_char)
                loss = (1 - LM_AUX_WEIGHT) * loss + LM_AUX_WEIGHT * criterion(fused, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
    return total / len(train_loader.dataset)

def evaluate(loader):
    model.eval()
    loss_sum, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            if LM_BETA > 0 and char_lm is not None:
                prev_idx = extract_prev_idx(x, NUM_CLASSES)
                if prev_idx is not None:
                    logits = fuse_batch_logits_with_lm(logits, char_lm, prev_idx, LM_BETA, idx_to_char)
            loss_sum += criterion(logits, y).item() * x.size(0)
            pred = logits.argmax(dim=-1)
            if y.ndim > 1:
                y = y.argmax(dim=-1)
            correct += (pred == y).sum().item()
            total += x.size(0)
    return loss_sum / total if total else 0.0, correct / total if total else 0.0

best_val_loss = float("inf")
patience_counter = 0
best_state = None

for epoch in range(EPOCHS):
    tl = train_epoch()
    vl, va = evaluate(val_loader)
    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d} | Train loss: {tl:.4f} | Val loss: {vl:.4f} | Val acc: {va:.4f}")

    # Early stopping: track best val loss
    if vl < best_val_loss:
        best_val_loss = vl
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch + 1} (no improvement for {EARLY_STOPPING_PATIENCE} epochs)")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

Epoch   1 | Train loss: 1.9185 | Val loss: 1.3341 | Val acc: 0.2018


In [ ]:
test_loss, test_acc = evaluate(test_loader)
print("=" * 60)
print(f"{MODEL.upper()} Results")
print(f"Test loss: {test_loss:.4f}  Test acc: {test_acc:.4f}")
print("=" * 60)

# Save model for inference_receiver.py (set simple_model.enabled: true and path/type in train_config.yaml)
save_dir = PROJECT_ROOT / "simple_models/models_trained"
save_dir.mkdir(exist_ok=True)
ckpt_path = save_dir / f"{MODEL}_classifier_simple.pt"
torch.save({"state_dict": model.state_dict(), "input_dim": dataset.input_dim}, ckpt_path)
print(f"Saved checkpoint to {ckpt_path}")

## Keyboard heatmap visualisation

In [ ]:
# Wrapper so compute_confusion_matrix gets fused logits (LM fusion)
class FusedModelWrapper(nn.Module):
    def __init__(self, model, lm, beta, idx_to_char):
        super().__init__()
        self.model = model
        self.lm = lm
        self.beta = beta
        self.idx_to_char = idx_to_char

    def forward(self, x):
        logits = self.model(x)
        if self.beta <= 0 or self.lm is None:
            return logits
        prev_idx = extract_prev_idx(x, NUM_CLASSES)
        if prev_idx is None:
            return logits
        return fuse_batch_logits_with_lm(logits, self.lm, prev_idx, self.beta, self.idx_to_char)

model_for_cm = FusedModelWrapper(model, char_lm, LM_BETA, idx_to_char)
cm_orig = compute_confusion_matrix_40x40(test_ds, model_for_cm, DEVICE)

# Keyboard heatmap for all chars in "hello world"
for ch in sorted(set("hello world")):
    plot_virtual_keyboard_heatmap(cm_orig, ch, 'Test')